# Fine-tune Qwen3.5-4B for the Decillion dsh agentic system (Kaggle, 2×T4)

QLoRA on both T4s (one process per GPU, DDP). Run the cells in order, with **GPU T4 ×2** and **Internet on**.

**Why QLoRA, not full fine-tuning.** Full fine-tuning a ~4.6B-parameter model needs >70 GB for fp32 master weights,
Adam state and gradients, even with DeepSpeed ZeRO-3 CPU offload. Kaggle gives 29 GB of RAM, 2×15 GB of VRAM and 20 GB
of `/kaggle/working`. A 4-bit base with LoRA adapters fits on one T4 per process, and checkpoints are small.

**The data is long.** Rendered with Qwen3.5's own chat template, rows are 6.8k–18.7k tokens (median 13.4k), because the
dsh system prompt plus tool schemas is about 11k tokens. `MAX_SEQ_LEN = 16384` keeps about 99% of rows. Only about 3% of
tokens are the assistant's, so the loss (and the 248k-vocabulary logits) is computed only at those positions.

**What gets loss.** Only the assistant turns after the last real user message: from after `<|im_start|>assistant\n`
through `<|im_end|>`, including tool calls. Qwen3.5's template keeps `<think>` reasoning only for those turns. Earlier
turns stay in the context exactly as the model sees them in production, but they are not trained, because their format
(no `<think>` block) differs from what inference produces.

**Checkpoints.** A checkpoint is written every 5% of an epoch (`SAVE_EVERY_EPOCH`), and only the newest 3
(`KEEP_CHECKPOINTS`) stay on disk; older ones are deleted after each save. Each holds the LoRA adapter, optimizer and
scheduler state (well under 1 GB), so the three together fit easily in `/kaggle/working`.

**Time.** A Kaggle session lasts at most 12 hours. Training stops itself with a checkpoint before `TIME_BUDGET_HOURS`.
Run the notebook again in a new session to resume, from this session's saved output (add it as an input) or from the
Hugging Face Hub (set `HF_REPO_ID` and a Kaggle secret `HF_TOKEN`). The smoke test (cell 7) measures the real speed on
the longest rows and tells you how many rows fit in the budget.

**Storage.** Only results go to `/kaggle/working` (adapter, checkpoints, merged model, metrics). The repository, the
built dataset, the vLLM environment and anything holding the API key stay under `/tmp/decillion`, which is not saved.

Cells: 1 hardware check · 2 install · 3 configuration · 4 build the dataset · 5 write the scripts · 6 choose and tokenize the mix ·
7 smoke test · 8 train · 9 metrics · 10 merge · 11 serve with vLLM · 12 Cloudflare tunnel ·
13 end-to-end API test.

Cells 2–10 are built from `colab/kaggle/` in the Aseman repository (`python3 colab/build_notebook.py`). Edit those files,
not this notebook.

In [ ]:
# 1. Hardware check. nvidia-smi is used instead of torch.cuda, so this kernel opens no CUDA context and holds no
# GPU memory while the training processes run.
import os, shutil, subprocess
import psutil

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # inherited by the training processes
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

gpus = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used,compute_cap', '--format=csv,noheader'],
                      capture_output=True, text=True).stdout.strip().splitlines()
for i, g in enumerate(gpus):
    print(f'GPU {i}: {g}')
NUM_GPUS = len(gpus)
if NUM_GPUS != 2:
    raise RuntimeError(f'Expected 2 GPUs, found {NUM_GPUS}. Choose Accelerator → GPU T4 ×2.')
ram = psutil.virtual_memory()
print(f'RAM: total {ram.total / 2**30:.1f} GiB, available {ram.available / 2**30:.1f} GiB, CPUs {os.cpu_count()}')
for path in ('/kaggle/working', '/tmp'):
    du = shutil.disk_usage(path)
    print(f'disk {path}: {du.free / 1e9:.0f} GB free of {du.total / 1e9:.0f} GB')

In [ ]:
# 2. Install the training stack, pinned, without touching Kaggle's CUDA build of PyTorch.
# A constraints file pins torch to the version that is already installed, so no package can upgrade it.
import subprocess, sys
torch_version = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'],
                               capture_output=True, text=True).stdout.strip()
open('/tmp/constraints.txt', 'w').write(f'torch=={torch_version}\n')
print('keeping torch', torch_version)
!pip install -c /tmp/constraints.txt transformers==5.24.0 peft==0.22.0 accelerate==1.16.0 datasets==5.2.0 \
    bitsandbytes==0.50.2 huggingface_hub==2.4.0 safetensors==0.8.0 jsonschema openai requests psutil

# Optional fast kernels for the Gated DeltaNet (linear attention) layers, 24 of the 32. Without them Transformers uses
# its reference PyTorch implementation: correct, but slower. Triton kernels are not guaranteed on Turing (T4, sm75),
# so they are tested here on the GPU and removed again if they fail or disagree.
INSTALL_FAST_KERNELS = True
if INSTALL_FAST_KERNELS:
    !pip install -q -c /tmp/constraints.txt flash-linear-attention==0.6.0
    test = r'''
import torch
from fla.ops.gated_delta_rule import chunk_gated_delta_rule, fused_recurrent_gated_delta_rule
torch.manual_seed(0)
B, T, H, K = 1, 512, 4, 128
dev, dt = 'cuda', torch.float16
q, k, v = (torch.randn(B, T, H, K, device=dev, dtype=dt, requires_grad=True) for _ in range(3))
g = (-torch.rand(B, T, H, device=dev) * 0.1).requires_grad_()
beta = torch.rand(B, T, H, device=dev, dtype=dt).requires_grad_()
o1, _ = chunk_gated_delta_rule(q, k, v, g=g, beta=beta, use_qk_l2norm_in_kernel=True)
o1.float().pow(2).mean().backward()
with torch.no_grad():
    o2, _ = fused_recurrent_gated_delta_rule(q, k, v, g=g, beta=beta, use_qk_l2norm_in_kernel=True)
err = (o1.float() - o2.float()).abs().max().item() / (o2.float().abs().max().item() + 1e-6)
grads_ok = all(t.grad is not None and torch.isfinite(t.grad).all() for t in (q, k, v, g, beta))
assert torch.isfinite(o1).all() and grads_ok and err < 2e-2, f'mismatch {err}, grads ok {grads_ok}'
print('fla kernels OK on', torch.cuda.get_device_name(0), f'(relative difference {err:.1e})')
'''
    open('/tmp/fla_test.py', 'w').write(test)
    r = subprocess.run([sys.executable, '/tmp/fla_test.py'], capture_output=True, text=True)
    print((r.stdout or '') + (r.stderr or '')[-1500:])
    if r.returncode != 0:
        print('fast kernels unusable here: uninstalling, training will use the PyTorch reference path')
        !pip uninstall -q -y flash-linear-attention fla-core
# Check what the training processes will actually import (pip errors are easy to miss above).
check = ("import torch, transformers, peft, bitsandbytes, accelerate, datasets; "
         "from transformers.models.auto.modeling_auto import MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES as M; "
         "print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, "
         "'| bitsandbytes', bitsandbytes.__version__, '| accelerate', accelerate.__version__); "
         "assert transformers.__version__ == '5.24.0' and 'qwen3_5' in M, 'transformers 5.24.0 is not installed'")
r = subprocess.run([sys.executable, '-c', check], capture_output=True, text=True)
print(r.stdout + r.stderr[-2000:])
if r.returncode != 0:
    raise RuntimeError('the pinned packages are not importable: read the pip output above (often a conflict with '
                       "Kaggle's torch). Fix the pin, rerun this cell.")
print('Restart the kernel only if pip reports that torch itself changed (it should not).')

In [ ]:
# 3. Configuration: the only cell you normally edit. It writes run_config.json, which every script reads.
import json, os, glob

MODEL_ID = 'Qwen/Qwen3.5-4B'
REPO = 'https://github.com/keyhmoham/Aseman.git'
BRANCH = 'main'                      # the branch that holds datasets/decillion-dsh-agentic

WORK = '/kaggle/working'             # saved with the notebook output (20 GB): results only
TMP = '/tmp/decillion'               # not saved: repository, dataset, vLLM, secrets
OUT_DIR = f'{WORK}/decillion-qwen35-4b'
CONFIG = f'{WORK}/run_config.json'
SCRIPTS = f'{WORK}/scripts'
REPO_DIR = f'{TMP}/Aseman'
DATA_DIR = f'{REPO_DIR}/datasets/decillion-dsh-agentic'
DIST_DIR = f'{TMP}/dist'
PREPARED_DIR = f'{TMP}/prepared'
MERGED_DIR = f'{OUT_DIR}/merged'
for d in (OUT_DIR, SCRIPTS, TMP):
    os.makedirs(d, exist_ok=True)

TOOLS_MODE = 'relevant'              # 'full' makes rows ~30k tokens: too long for a T4
MAX_SEQ_LEN = 16384                  # keeps ~99% of rows; rows that do not fit are dropped, never truncated
TRAIN_ROWS = 1500                    # rows per epoch; check the smoke test's estimate of what fits the time budget
SINGLE_TURN_SHARE = 0.5              # share of single-turn families (sampled evenly across families) vs sessions
EVAL_ROWS = 48
EPOCHS = 1

LOAD_IN_4BIT = True                  # fp16 base weights (~9 GB) leave too little of a T4 for 16k-token activations
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.0
LEARNING_RATE = 1e-4
GRAD_ACCUM = 8                       # x 2 GPUs x 1 row = 16 rows per optimizer step
WARMUP_RATIO = 0.05
SAVE_EVERY_EPOCH = 0.05              # write a checkpoint every 5% of an epoch (rounded to whole optimizer steps)
KEEP_CHECKPOINTS = 3                 # only the newest 3 stay on disk; older ones are deleted after each save
EVAL_EVERY = 25
TIME_BUDGET_HOURS = 9.5              # training stops with a checkpoint after this; leaves time to merge and save
SEED = 3407

# Resuming in a new session: either attach this notebook's earlier output as an input (found automatically
# below), or set HF_REPO_ID (a private model repo) and add a Kaggle secret HF_TOKEN: checkpoints are pushed there.
HF_REPO_ID = ''                      # e.g. 'your-name/decillion-qwen35-4b-lora'
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception:
    print('no HF_TOKEN secret: downloads are anonymous' + (' and HF_REPO_ID cannot be used' if HF_REPO_ID else ''))
    if HF_REPO_ID:
        raise RuntimeError('HF_REPO_ID is set but the Kaggle secret HF_TOKEN is missing')
earlier = sorted(glob.glob('/kaggle/input/**/decillion-qwen35-4b/checkpoints', recursive=True))
RESUME_FROM = earlier[-1] if earlier else None
print('resume from attached input:', RESUME_FROM)

cfg = dict(model_id=MODEL_ID, out_dir=OUT_DIR, dist_dir=DIST_DIR, prepared_dir=PREPARED_DIR, merged_dir=MERGED_DIR,
           max_seq_len=MAX_SEQ_LEN, train_rows=TRAIN_ROWS, single_turn_share=SINGLE_TURN_SHARE, eval_rows=EVAL_ROWS,
           epochs=EPOCHS, load_in_4bit=LOAD_IN_4BIT, lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
           learning_rate=LEARNING_RATE, grad_accum=GRAD_ACCUM, warmup_ratio=WARMUP_RATIO, save_every_epoch=SAVE_EVERY_EPOCH,
           keep_checkpoints=KEEP_CHECKPOINTS,
           eval_every=EVAL_EVERY, time_budget_hours=TIME_BUDGET_HOURS, seed=SEED, hf_repo_id=HF_REPO_ID,
           resume_from=RESUME_FROM, num_proc=max(1, (os.cpu_count() or 2)))
json.dump(cfg, open(CONFIG, 'w'), indent=1)
print(json.dumps(cfg, indent=1))

import re, subprocess, sys
LOGS = f'{OUT_DIR}/logs'
os.makedirs(LOGS, exist_ok=True)

def run(cmd, name):
    """Run a shell command, stream its output here and into logs/<name>.log. On failure, show the first real error
    (the Python traceback from whichever process crashed) instead of whatever the next line trips over."""
    log_path = f'{LOGS}/{name}.log'
    for old in glob.glob(f'{OUT_DIR}/error_rank*.txt'):
        os.remove(old)
    print('$', cmd, '\n  log →', log_path, flush=True)
    lines = []
    with open(log_path, 'w') as log, subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                                      stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            log.write(line)
            lines.append(line)
            if not re.search(r'\d+%\|', line):      # keep progress bars out of the notebook output
                print(line, end='', flush=True)
    if p.returncode == 0:
        return
    errors = sorted(glob.glob(f'{OUT_DIR}/error_rank*.txt'))
    if errors:
        detail = open(errors[0]).read()
    else:
        text = ''.join(lines)
        start = text.find('Traceback (most recent call last)')
        detail = text[start:start + 6000] if start >= 0 else text[-6000:]
    print('\n' + '=' * 30 + f' {name} FAILED (exit {p.returncode}) ' + '=' * 30)
    print(detail)
    if 'OutOfMemoryError' in detail or 'CUDA out of memory' in detail:
        print('Out of GPU memory: lower MAX_SEQ_LEN in cell 3 (e.g. 12288), then rerun cells 3, 6 and this one.')
    raise RuntimeError(f'{name} failed (exit {p.returncode}); the error is printed above, full log in {log_path}')

In [ ]:
# 4. Get the dataset generator from GitHub and build the training rows (about 3 minutes).
import os
if not os.path.exists(REPO_DIR):
    !git clone -q --depth 1 -b {BRANCH} {REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q --ff-only
!python3 {DATA_DIR}/scripts/build.py --tools {TOOLS_MODE} --out {DIST_DIR}
!ls -la {DIST_DIR}

### 5. Write the helper scripts

The next three cells write `prepare_data.py`, `train_lora.py` and `merge_lora.py` to `/kaggle/working/scripts`. They are generated from `colab/kaggle/` in the repository, where they can be tested on their own; every script reads its settings from `run_config.json` (cell 3), so there is a single source of truth.

In [ ]:
%%writefile /kaggle/working/scripts/prepare_data.py
#!/usr/bin/env python3
"""Pick the training mix, render it with the model's chat template and write token ids + loss labels.

Run as its own process (not inside the notebook kernel) so the ~1 GB train file and the rendered text
never stay resident in the kernel while the GPUs train:

    python3 prepare_data.py --config /kaggle/working/run_config.json

What it does:
  * Two streaming passes over train.jsonl.gz: the first records only (line number, category), the second
    keeps just the selected lines, so memory stays small.
  * Single-turn families are sampled round-robin by category (rare families are not drowned out);
    multi-turn sessions are sampled at random. The mix is set by train_rows / single_turn_share.
  * Each row is rendered with the tokenizer's own chat template (tools included) and tokenized once.
    Rows longer than max_seq_len are dropped, never truncated (truncation would cut the answer off).
  * Loss labels cover only the assistant turns after the last real user message, from just after
    "<|im_start|>assistant\\n" through "<|im_end|>". Qwen3.5's template renders <think> only for those turns
    and strips the reasoning (and the <think> block) from earlier ones, so training earlier turns would
    teach a format that inference (which always opens with "<think>\\n") never uses. Earlier turns stay in
    the context, exactly as the model will see them in production.
"""
import argparse, gzip, json, os, random, re, sys, time
from collections import Counter, defaultdict
from multiprocessing import Pool

IM_START, IM_END = '<|im_start|>', '<|im_end|>'
USER_HDR, ASSIST_HDR = IM_START + 'user\n', IM_START + 'assistant\n'
CAT = re.compile(r'"category":\s*"([^"]+)"')


def assistant_spans(text):
    """Character spans of the assistant turns that should carry loss (see module docstring)."""
    starts = [m.start() for m in re.finditer(re.escape(IM_START), text)]
    last_query = -1
    for p in starts:
        if text.startswith(USER_HDR, p) and not text.startswith('<tool_response>', p + len(USER_HDR)):
            last_query = p
    spans = []
    for p in starts:
        if p > last_query and text.startswith(ASSIST_HDR, p):
            b = p + len(ASSIST_HDR)
            e = text.find(IM_END, b)
            if e < 0:
                continue
            spans.append((b, e + len(IM_END)))
    return spans


def encode_text(text, tok, max_len):
    """-> (input_ids, labels, reason). reason is None when the row is usable."""
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = enc['input_ids']
    if len(ids) > max_len:
        return None, None, 'too_long'
    spans = assistant_spans(text)
    labels = [-100] * len(ids)
    j = 0
    for k, (s, e) in enumerate(enc['offset_mapping']):
        while j < len(spans) and spans[j][1] <= s:
            j += 1
        if j < len(spans) and e > s and s >= spans[j][0] and e <= spans[j][1]:
            labels[k] = ids[k]
    if not any(l != -100 for l in labels):
        return None, None, 'no_assistant_tokens'
    return ids, labels, None


_TOK = None
_MAX = None


def _init(model_id, max_len):
    global _TOK, _MAX
    from transformers import AutoTokenizer
    _TOK = AutoTokenizer.from_pretrained(model_id)
    _MAX = max_len


def _work(line):
    try:
        row = json.loads(line)
        text = _TOK.apply_chat_template(row['messages'], tools=row.get('tools'), tokenize=False)
        ids, labels, reason = encode_text(text, _TOK, _MAX)
    except Exception as exc:  # a broken row must not kill the whole run
        return {'reason': 'error', 'error': f'{type(exc).__name__}: {exc}'[:300]}
    if reason:
        return {'reason': reason}
    return {'reason': None, 'id': row.get('id', ''), 'category': row.get('category', ''),
            'input_ids': ids, 'labels': labels}


def round_robin(by_cat, rng):
    """Interleave categories so any prefix of the result is as balanced as possible."""
    pools = {c: rng.sample(v, len(v)) for c, v in by_cat.items()}
    cats = sorted(pools)
    rng.shuffle(cats)
    out = []
    while any(pools.values()):
        for c in cats:
            if pools[c]:
                out.append(pools[c].pop())
    return out


def scan_categories(path):
    cats = []
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for line in f:
            m = CAT.search(line[:400])
            cats.append(m.group(1) if m else '?')
    return cats


def read_lines(path, wanted):
    wanted = set(wanted)
    got = {}
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i in wanted:
                got[i] = line
    return got


def process(order, lines, quota, pool, label):
    """Encode candidates in `order` until `quota` usable rows are collected."""
    kept, reasons, errors = [], Counter(), []
    pos, chunk = 0, max(64, quota // 4)
    t0 = time.time()
    while len(kept) < quota and pos < len(order):
        batch = order[pos:pos + chunk]
        pos += chunk
        for res in pool.imap(_work, (lines[i] for i in batch), chunksize=4):
            if res['reason']:
                reasons[res['reason']] += 1
                if res['reason'] == 'error' and len(errors) < 5:
                    errors.append(res['error'])
                continue
            if len(kept) < quota:
                kept.append(res)
        print(f'  {label}: {len(kept)}/{quota} kept, {sum(reasons.values())} dropped '
              f'({dict(reasons)}), {time.time() - t0:.0f}s', flush=True)
    for e in errors:
        print('  error:', e)
    return kept, reasons


def save(rows, path):
    from datasets import Dataset, Features, Sequence, Value
    feats = Features({'id': Value('string'), 'category': Value('string'),
                      'input_ids': Sequence(Value('int32')), 'labels': Sequence(Value('int32'))})
    cols = {k: [r[k] for r in rows] for k in feats}
    Dataset.from_dict(cols, features=feats).save_to_disk(path)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--config', required=True)
    cfg = json.load(open(ap.parse_args().config))
    rng = random.Random(cfg['seed'])
    train_path = os.path.join(cfg['dist_dir'], 'train.jsonl.gz')
    val_path = os.path.join(cfg['dist_dir'], 'validation.jsonl.gz')
    max_len, n_train = cfg['max_seq_len'], cfg['train_rows']
    n_single = round(n_train * cfg['single_turn_share'])
    n_multi = n_train - n_single
    over = 1.25  # candidates beyond the quota, to replace rows that are too long

    print('scanning categories…', flush=True)
    cats = scan_categories(train_path)
    single, multi = defaultdict(list), []
    for i, c in enumerate(cats):
        (multi.append(i) if c.startswith('session.') else single[c].append(i))
    single_order = round_robin(single, rng)
    multi_order = rng.sample(multi, len(multi))
    if len(single_order) < n_single:  # not enough single-turn rows: give the rest to sessions
        n_multi += n_single - len(single_order)
        n_single = len(single_order)
    single_cand = single_order[:int(n_single * over) + 8]
    multi_cand = multi_order[:int(n_multi * over) + 8]
    print(f'{len(cats)} train rows: {sum(map(len, single.values()))} single-turn in {len(single)} '
          f'families, {len(multi)} sessions; want {n_single} single + {n_multi} sessions', flush=True)
    lines = read_lines(train_path, single_cand + multi_cand)

    val_by_cat = defaultdict(list)
    val_lines = {}
    with gzip.open(val_path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            m = CAT.search(line[:400])
            val_by_cat[m.group(1) if m else '?'].append(i)
            val_lines[i] = line
    val_order = round_robin(val_by_cat, rng)

    with Pool(cfg.get('num_proc') or os.cpu_count() or 2, initializer=_init,
              initargs=(cfg['model_id'], max_len)) as pool:
        s_rows, s_why = process(single_cand, lines, n_single, pool, 'single-turn')
        m_rows, m_why = process(multi_cand, lines, n_multi, pool, 'sessions')
        e_rows, e_why = process(val_order, val_lines, cfg['eval_rows'], pool, 'validation')

    train = s_rows + m_rows
    rng.shuffle(train)
    if not train or not e_rows:
        sys.exit(f'no usable rows (train {len(train)}, eval {len(e_rows)}): check max_seq_len={max_len} '
                 f'and the drop reasons above')
    out = cfg['prepared_dir']
    os.makedirs(out, exist_ok=True)
    save(train, os.path.join(out, 'train'))
    save(e_rows, os.path.join(out, 'eval'))

    def summary(rows):
        lens = sorted(len(r['input_ids']) for r in rows)
        lab = sum(sum(1 for x in r['labels'] if x != -100) for r in rows)
        return {'rows': len(rows), 'tokens': sum(lens), 'label_tokens': lab,
                'len_min': lens[0], 'len_median': lens[len(lens) // 2], 'len_max': lens[-1]}

    stats = {'max_seq_len': max_len, 'train': summary(train), 'eval': summary(e_rows),
             'dropped': {'single': dict(s_why), 'sessions': dict(m_why), 'eval': dict(e_why)},
             'train_categories': dict(Counter(r['category'] for r in train).most_common())}
    json.dump(stats, open(os.path.join(out, 'stats.json'), 'w'), indent=1)
    print(json.dumps({k: stats[k] for k in ('train', 'eval', 'dropped')}, indent=1))
    if len(train) < n_train:
        print(f'WARNING: only {len(train)} of {n_train} wanted training rows fit in {max_len} tokens')


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/scripts/train_lora.py
#!/usr/bin/env python3
"""QLoRA fine-tune of Qwen3.5-4B on the prepared Decillion dsh rows, one process per GPU (DDP).

    torchrun --standalone --nproc_per_node=2 train_lora.py --config /kaggle/working/run_config.json [--smoke]

Why this shape (2x T4, 15 GB each, 29 GB host RAM, 20 GB /kaggle/working, 12 h sessions):
  * Full fine-tuning does not fit: fp32 master weights + Adam + grads for ~4.6B params is >70 GB even with
    ZeRO-3 CPU offload. A 4-bit base with LoRA adapters fits in one T4 per process.
  * The rows are 7k-19k tokens but only ~3% of them are assistant tokens. Logits over the 248k vocabulary
    for every position would be 8-16 GB per row, so the loss is computed only at labelled positions, in
    chunks, with the lm_head recomputed in backward (peak <1 GB).
  * A time budget stops training with a saved checkpoint before Kaggle kills the session; the next session
    resumes from it (local, a Kaggle input dataset, or the Hugging Face Hub).
  * --smoke runs two training steps on the LONGEST rows to prove memory fits before a long run.
"""
import argparse, glob, json, math, os, sys, time

import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from datasets import load_from_disk
from peft import LoraConfig, get_peft_model
from transformers import (AutoModelForImageTextToText, AutoTokenizer, BitsAndBytesConfig, Trainer,
                          TrainerCallback, TrainingArguments)

# LoRA only on the language model: full-attention, Gated DeltaNet (linear attention) and MLP projections.
# The vision tower is never trained (this dataset is text-only) and stays in the checkpoint for vLLM.
LORA_TARGETS = (r'.*language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)'
                r'|linear_attn\.(in_proj_qkv|in_proj_z|out_proj)|mlp\.(gate_proj|up_proj|down_proj))')
CE_CHUNK = 512  # labelled positions per lm_head chunk


# ---------------------------------------------------------------------------------------------------------------
# Attention that fits a T4 at 16k tokens.
# On Turing there is no FlashAttention, and for Qwen3.5's full-attention layers (head_dim 256, grouped-query
# attention) PyTorch's SDPA falls back to the "math" kernel, which materializes the whole [heads, L, L] score
# matrix in fp32: 16 x 16k x 16k x 4 B = 16 GB for a single layer. This implementation uses the memory-efficient
# kernel when a probe on this GPU shows it can run forward AND backward at this head size, and otherwise computes
# attention in query blocks, each recomputed in backward, so at most one block of scores (~1 GB at 16k) exists.
# ---------------------------------------------------------------------------------------------------------------
ATTN_IMPL = 't4_blocked'
ATTN_BLOCK = 1024
_EFFICIENT_OK = {}


def _repeat_kv(x, n):
    if n == 1:
        return x
    b, h, l, d = x.shape
    return x[:, :, None].expand(b, h, n, l, d).reshape(b, h * n, l, d)


def probe_attention(device, dtype, head_dim):
    """Decide ONCE, before training, whether the memory-efficient SDPA kernel can run forward and backward here.

    This must never run inside the model's forward: under activation checkpointing that forward is replayed in
    backward, and a probe that ran only the first time changes what autograd saved (CheckpointError).
    """
    key = (str(device), dtype, head_dim)
    ok = False
    if device.type == 'cuda':
        try:
            import warnings
            from torch.nn.attention import SDPBackend, sdpa_kernel
            with warnings.catch_warnings(), torch.enable_grad():
                warnings.simplefilter('ignore')
                q, k, v = (torch.randn(1, 2, 256, head_dim, device=device, dtype=dtype, requires_grad=True)
                           for _ in range(3))
                with sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION]):
                    F.scaled_dot_product_attention(q, k, v, is_causal=True).float().sum().backward()
                ok = all(bool(torch.isfinite(t.grad).all()) for t in (q, k, v))
            torch.cuda.synchronize(device)
        except Exception:
            ok = False
    _EFFICIENT_OK[key] = ok
    log(f'attention for head_dim {head_dim} ({dtype}) on {device}: '
        + ('memory-efficient SDPA kernel' if ok else f'query-blocked attention ({ATTN_BLOCK} queries per block)'))
    return ok


def _efficient_kernel_works(device, dtype, head_dim):
    # Lookup only (see probe_attention). Anything not probed uses the blocked path, which always works.
    return _EFFICIENT_OK.get((str(device), dtype, head_dim), False)


def _attn_block(q, k, v, mask, q0, causal, scaling):
    """Scores for queries [q0, q0+bq) against keys [0, kend) only (causal), softmax in fp32."""
    bq = q.shape[2]
    kend = min(k.shape[2], q0 + bq) if causal else k.shape[2]
    k, v = k[:, :, :kend], v[:, :, :kend]
    scores = torch.matmul(q.float(), k.float().transpose(-1, -2)) * scaling
    if mask is not None:
        m = mask[..., q0:q0 + bq, :kend]
        scores = scores.masked_fill(~m, float('-inf')) if m.dtype == torch.bool else scores + m.float()
    if causal:
        qi = torch.arange(q0, q0 + bq, device=q.device)[:, None]
        kj = torch.arange(kend, device=q.device)[None, :]
        scores = scores.masked_fill(kj > qi, float('-inf'))
    probs = torch.softmax(scores, dim=-1).nan_to_num(0.0)  # fully masked rows (padding) -> 0
    return torch.matmul(probs.to(v.dtype), v)


def t4_attention_forward(module, query, key, value, attention_mask, dropout=0.0, scaling=None, is_causal=None,
                         **kwargs):
    n_rep = query.shape[1] // key.shape[1]
    key, value = _repeat_kv(key, n_rep), _repeat_kv(value, n_rep)
    scaling = scaling if scaling is not None else query.shape[-1] ** -0.5
    causal = (is_causal if is_causal is not None else getattr(module, 'is_causal', True)) and query.shape[2] > 1
    if attention_mask is not None and attention_mask.dim() == 4 and attention_mask.shape[-1] > key.shape[2]:
        attention_mask = attention_mask[..., :key.shape[2]]
    if _efficient_kernel_works(query.device, query.dtype, query.shape[-1]):
        from torch.nn.attention import SDPBackend, sdpa_kernel
        with sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION]):
            out = F.scaled_dot_product_attention(query, key, value, attn_mask=attention_mask,
                                                 is_causal=causal and attention_mask is None, scale=scaling)
    else:
        # Causal masking is applied by position inside each block, so a 4D mask is only needed for padding.
        blocks = []
        for q0 in range(0, query.shape[2], ATTN_BLOCK):
            qb = query[:, :, q0:q0 + ATTN_BLOCK]
            if torch.is_grad_enabled():
                blocks.append(checkpoint(_attn_block, qb, key, value, attention_mask, q0, causal, scaling,
                                         use_reentrant=False))
            else:
                blocks.append(_attn_block(qb, key, value, attention_mask, q0, causal, scaling))
        out = torch.cat(blocks, dim=2)
    return out.transpose(1, 2).contiguous(), None


def register_t4_attention():
    from transformers import AttentionInterface
    from transformers.masking_utils import ALL_MASK_ATTENTION_FUNCTIONS, sdpa_mask
    AttentionInterface.register(ATTN_IMPL, t4_attention_forward)
    ALL_MASK_ATTENTION_FUNCTIONS.register(ATTN_IMPL, sdpa_mask)  # None when there is no padding


def rank0():
    return int(os.environ.get('RANK', '0')) == 0


def log(*a):
    if rank0():
        print(*a, flush=True)


class Collator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, feats):
        n = max(len(f['input_ids']) for f in feats)
        ids = torch.full((len(feats), n), self.pad_id, dtype=torch.long)
        labels = torch.full((len(feats), n), -100, dtype=torch.long)
        mask = torch.zeros((len(feats), n), dtype=torch.long)
        for b, f in enumerate(feats):
            k = len(f['input_ids'])
            ids[b, :k] = torch.tensor(f['input_ids'])
            labels[b, :k] = torch.tensor(f['labels'])
            mask[b, :k] = 1
        return {'input_ids': ids, 'attention_mask': mask, 'labels': labels}


def _chunk_ce(h, weight, target):
    # Runs outside the Trainer's autocast (which wraps only the model's forward), so match dtypes explicitly.
    logits = F.linear(h.to(weight.dtype), weight).float()
    return F.cross_entropy(logits, target, reduction='sum')


def selected_ce(hidden, weight, target):
    """Summed cross entropy of hidden[n, d] @ weight.T against target[n], chunked and recomputed."""
    total = hidden.new_zeros((), dtype=torch.float32)
    for s in range(0, hidden.shape[0], CE_CHUNK):
        total = total + checkpoint(_chunk_ce, hidden[s:s + CE_CHUNK], weight, target[s:s + CE_CHUNK],
                                   use_reentrant=False)
    return total


class AssistantLossTrainer(Trainer):
    """Mean cross entropy over each row's assistant tokens; logits only where there is a label."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs['labels']
        out = model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'],
                    use_cache=False, output_hidden_states=True, logits_to_keep=1)
        hidden = out.hidden_states[-1]                      # final-norm output, [B, L, d]
        target = labels[:, 1:]
        keep = target != -100
        h = hidden[:, :-1][keep]                            # positions whose NEXT token is a label
        t = target[keep]
        lm_head = self.accelerator.unwrap_model(model).get_output_embeddings()
        if t.numel() == 0:
            loss = hidden.sum() * 0.0
        else:
            loss = selected_ce(h, lm_head.weight, t) / t.numel()
        return (loss, {'loss': loss}) if return_outputs else loss


class TimeBudget(TrainerCallback):
    """Stop (with a checkpoint) before the session limit. All ranks decide together, or DDP would hang."""

    def __init__(self, seconds):
        self.deadline = time.time() + seconds
        self.hit = False

    def on_step_end(self, args, state, control, **kw):
        flag = torch.tensor([1.0 if time.time() > self.deadline else 0.0],
                            device='cuda' if torch.cuda.is_available() else 'cpu')
        if dist.is_available() and dist.is_initialized():
            dist.all_reduce(flag, op=dist.ReduceOp.MAX)
        if flag.item() > 0 and not self.hit:
            self.hit = True
            control.should_save = True
            control.should_training_stop = True
            log(f'time budget reached at step {state.global_step}: saving a checkpoint and stopping')
        return control


class NanGuard(TrainerCallback):
    """Logged losses are gathered across ranks, so every rank sees the same value and stops together."""

    def __init__(self, patience=3):
        self.bad, self.patience = 0, patience

    def on_log(self, args, state, control, logs=None, **kw):
        loss = (logs or {}).get('loss')
        if loss is None:
            return control
        self.bad = self.bad + 1 if not math.isfinite(loss) else 0
        if self.bad >= self.patience:
            log(f'loss is not finite for {self.bad} logs in a row: stopping. Lower the learning rate, or '
                'check the fp16 overflow notes in the notebook.')
            control.should_training_stop = True
        return control


def checkpointing_args():
    """Transformers renamed gradient_checkpointing to activation_checkpointing; accept either."""
    import dataclasses
    names = {f.name for f in dataclasses.fields(TrainingArguments)}
    key = 'activation_checkpointing' if 'activation_checkpointing' in names else 'gradient_checkpointing'
    return {key: True, f'{key}_kwargs': {'use_reentrant': False}}


def check_versions():
    import transformers, peft
    from transformers.models.auto.modeling_auto import MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES as M
    log(f'torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | '
        f'GPUs {torch.cuda.device_count()}')
    if 'qwen3_5' not in M:
        raise RuntimeError(f'transformers {transformers.__version__} does not know Qwen3.5: cell 2 did not install '
                           'the pinned version. Rerun cell 2 and read its pip output (it is no longer quiet).')


def latest_checkpoint(*dirs):
    found = []
    for d in dirs:
        if d and os.path.isdir(d):
            found += [p for p in glob.glob(os.path.join(d, 'checkpoint-*'))
                      if os.path.exists(os.path.join(p, 'trainer_state.json'))]
            if os.path.exists(os.path.join(d, 'trainer_state.json')):
                found.append(d)  # a "last-checkpoint" folder from the Hub, or a checkpoint path itself
    if not found:
        return None
    return max(found, key=lambda p: json.load(open(os.path.join(p, 'trainer_state.json')))['global_step'])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--config', required=True)
    ap.add_argument('--smoke', action='store_true', help='2 steps on the longest rows, then stop')
    opt = ap.parse_args()
    cfg = json.load(open(opt.config))
    check_versions()
    register_t4_attention()
    cuda = torch.cuda.is_available()
    local_rank = int(os.environ.get('LOCAL_RANK', '0'))
    world = int(os.environ.get('WORLD_SIZE', '1'))
    if cuda:
        torch.cuda.set_device(local_rank)
    # torch.cuda.is_bf16_supported() also counts emulation and returns True on a T4: ask for real bf16 hardware.
    native_bf16 = cuda and torch.cuda.get_device_capability()[0] >= 8
    compute_dtype = torch.bfloat16 if native_bf16 else (torch.float16 if cuda else torch.float32)
    log(f'precision: {compute_dtype} ({"native bf16" if native_bf16 else "fp16 with loss scaling" if cuda else "cpu"})')

    out_dir = cfg['out_dir']
    ckpt_dir = os.path.join(out_dir, 'smoke' if opt.smoke else 'checkpoints')
    adapter_dir = os.path.join(out_dir, 'adapter')

    tok = AutoTokenizer.from_pretrained(cfg['model_id'])
    train_ds = load_from_disk(os.path.join(cfg['prepared_dir'], 'train'))
    eval_ds = load_from_disk(os.path.join(cfg['prepared_dir'], 'eval'))
    if opt.smoke:  # worst case for memory: the longest rows
        lens = [len(x) for x in train_ds['input_ids']]
        top = sorted(range(len(lens)), key=lambda i: -lens[i])[:2 * world]
        train_ds = train_ds.select(top)
        eval_ds = eval_ds.select(range(min(world, len(eval_ds))))
        log('smoke test on rows of', [lens[i] for i in top], 'tokens')

    steps_per_epoch = math.ceil(len(train_ds) / (world * cfg['grad_accum']))
    total_steps = 2 if opt.smoke else steps_per_epoch * cfg['epochs']
    # A checkpoint every `save_every_epoch` of an epoch (0.05 = 20 per epoch), in optimizer steps; only the newest
    # `keep_checkpoints` stay on disk (the Trainer deletes older ones after each save).
    save_steps = max(1, round(steps_per_epoch * cfg.get('save_every_epoch', 0.05)))
    keep_ckpt = max(1, int(cfg.get('keep_checkpoints', 3)))
    eval_steps = max(1, cfg['eval_every'])
    args = TrainingArguments(
        output_dir=ckpt_dir,
        per_device_train_batch_size=1,          # rows are 7k-16k tokens; padding a batch would waste compute
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=1 if opt.smoke else cfg['grad_accum'],
        num_train_epochs=cfg['epochs'],
        max_steps=total_steps,
        learning_rate=cfg['learning_rate'],
        lr_scheduler_type='cosine',
        warmup_steps=0 if opt.smoke else max(1, math.ceil(total_steps * cfg['warmup_ratio'])),
        weight_decay=0.0,
        optim='adamw_torch',
        max_grad_norm=1.0,
        fp16=cuda and not native_bf16,          # T4: fp16 (bf16 there is only emulated, and slow)
        bf16=native_bf16,
        **checkpointing_args(),
        logging_steps=1 if opt.smoke else 5,
        logging_first_step=True,
        eval_strategy='no' if opt.smoke else 'steps',
        eval_steps=eval_steps,
        save_strategy='no' if opt.smoke else 'steps',
        save_steps=save_steps,
        save_total_limit=keep_ckpt,
        prediction_loss_only=True,
        label_names=['labels'],
        remove_unused_columns=False,
        dataloader_num_workers=2,
        dataloader_pin_memory=cuda,
        ddp_find_unused_parameters=False,
        ddp_backend=None if cuda else 'gloo',  # gloo only for CPU dry runs
        report_to='none',
        seed=cfg['seed'],
        data_seed=cfg['seed'],
        push_to_hub=bool(cfg.get('hf_repo_id')) and not opt.smoke,
        hub_model_id=cfg.get('hf_repo_id') or None,
        hub_strategy='checkpoint',
        hub_private_repo=True,
    )

    quant = None
    if cuda and cfg['load_in_4bit']:
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                   bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
                                   llm_int8_skip_modules=['visual', 'lm_head'])
    model = AutoModelForImageTextToText.from_pretrained(
        cfg['model_id'],
        dtype=compute_dtype,
        quantization_config=quant,
        device_map={'': local_rank} if cuda else None,
        attn_implementation=ATTN_IMPL,
    )
    model.config.use_cache = False
    if hasattr(model.config, 'text_config'):
        model.config.text_config.use_cache = False
    model.requires_grad_(False)
    text_cfg = getattr(model.config, 'text_config', model.config)
    head_dim = getattr(text_cfg, 'head_dim', None) or text_cfg.hidden_size // text_cfg.num_attention_heads
    probe_attention(torch.device('cuda', local_rank) if cuda else torch.device('cpu'), compute_dtype, head_dim)
    model = get_peft_model(model, LoraConfig(
        r=cfg['lora_r'], lora_alpha=cfg['lora_alpha'], lora_dropout=cfg['lora_dropout'],
        target_modules=LORA_TARGETS, bias='none'))
    if rank0():
        model.print_trainable_parameters()

    trainer = AssistantLossTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
        data_collator=Collator(tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id),
        processing_class=tok,
        callbacks=[TimeBudget(cfg['time_budget_hours'] * 3600), NanGuard()],
    )
    # compute_loss already returns a per-row mean: let the Trainer divide by the accumulation steps.
    trainer.model_accepts_loss_kwargs = False
    if hasattr(trainer, 'loss_is_scaled_for_ga'):
        trainer.loss_is_scaled_for_ga = False

    resume = None if opt.smoke else latest_checkpoint(ckpt_dir, cfg.get('resume_from'))
    log(f'train rows {len(train_ds)} | eval rows {len(eval_ds)} | world {world} | '
        f'optimizer steps {total_steps} | checkpoint every {save_steps} steps '
        f'({save_steps / steps_per_epoch:.3f} epoch), keeping the last {keep_ckpt} | resume from {resume}')
    t0 = time.time()
    trainer.train(resume_from_checkpoint=resume)
    took = time.time() - t0

    if opt.smoke:
        mem = torch.cuda.max_memory_allocated() / 2**30 if cuda else 0.0
        peak = torch.tensor([mem], device='cuda' if cuda else 'cpu')
        if dist.is_available() and dist.is_initialized():
            dist.all_reduce(peak, op=dist.ReduceOp.MAX)
        losses = [h['loss'] for h in trainer.state.log_history if 'loss' in h]
        ok = bool(losses) and all(math.isfinite(x) for x in losses)
        if rank0():
            sec_per_row = took / max(1, len(train_ds))
            rows_budget = int(cfg['time_budget_hours'] * 3600 / sec_per_row) if sec_per_row else 0
            res = {'ok': ok, 'losses': losses, 'peak_gib': round(peak.item(), 2),
                   'seconds': round(took, 1), 'sec_per_row_all_gpus': round(sec_per_row, 1),
                   'rows_that_fit_time_budget': rows_budget}
            json.dump(res, open(os.path.join(out_dir, 'smoke.json'), 'w'), indent=1)
            print('SMOKE', json.dumps(res), flush=True)
        if not ok:
            sys.exit(1)
        return

    complete = trainer.state.global_step >= total_steps
    metrics = trainer.evaluate()
    if 'eval_loss' in metrics:
        metrics['eval_perplexity'] = math.exp(min(float(metrics['eval_loss']), 20))
    trainer.save_model(adapter_dir)  # the LoRA adapter only (small); rank 0 writes
    if rank0():
        tok.save_pretrained(adapter_dir)
        status = {'complete': complete, 'global_step': trainer.state.global_step, 'max_steps': total_steps,
                  'hours_this_session': round(took / 3600, 2), 'eval': metrics,
                  'log_history': trainer.state.log_history[-50:]}
        json.dump(status, open(os.path.join(out_dir, 'train_status.json'), 'w'), indent=1)
        json.dump(metrics, open(os.path.join(out_dir, 'eval_metrics.json'), 'w'), indent=1)
        print('adapter →', adapter_dir, '| complete' if complete else
              f'| stopped at step {trainer.state.global_step}/{total_steps}: run the training cell again in '
              'a new session to resume', flush=True)
        print('eval:', metrics, flush=True)
    if cfg.get('hf_repo_id'):
        trainer.push_to_hub(commit_message='adapter' if complete else 'partial adapter')


if __name__ == '__main__':
    try:
        main()
    except BaseException:
        import traceback
        rank = os.environ.get('RANK', '0')
        msg = traceback.format_exc()
        try:
            cfg_path = sys.argv[sys.argv.index('--config') + 1]
            path = os.path.join(json.load(open(cfg_path))['out_dir'], f'error_rank{rank}.txt')
            open(path, 'w').write(msg)
        except Exception:
            pass
        print(f'[rank {rank}] FAILED\n{msg}', file=sys.stderr, flush=True)
        raise

In [ ]:
%%writefile /kaggle/working/scripts/merge_lora.py
#!/usr/bin/env python3
"""Merge the trained LoRA adapter into a 16-bit copy of the base model, for vLLM.

    python3 merge_lora.py --config /kaggle/working/run_config.json

Runs on the CPU (the base model in fp16 is ~9 GB of host RAM), so it works right after training without
touching GPU memory. The adapter is applied to the full-precision base, not to the 4-bit one it was
trained against (the standard QLoRA export). The result keeps the full Qwen3.5 architecture, vision tower
included, so vLLM loads it like the base model. Processor and chat-template files are copied from the base.
"""
import argparse, json, os, shutil, sys

import torch
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForImageTextToText, AutoProcessor


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--config', required=True)
    ap.add_argument('--adapter', help='adapter folder (default: <out_dir>/adapter)')
    opt = ap.parse_args()
    cfg = json.load(open(opt.config))
    adapter = opt.adapter or os.path.join(cfg['out_dir'], 'adapter')
    merged = cfg['merged_dir']
    if not os.path.exists(os.path.join(adapter, 'adapter_config.json')):
        sys.exit(f'no adapter at {adapter}: train first')
    status_path = os.path.join(cfg['out_dir'], 'train_status.json')
    if not opt.adapter and os.path.exists(status_path) and not json.load(open(status_path)).get('complete'):
        print('WARNING: training stopped early (time budget); merging the partial adapter')

    parent = os.path.dirname(merged.rstrip('/')) or '.'
    os.makedirs(parent, exist_ok=True)
    free_gb = shutil.disk_usage(parent).free / 1e9
    if free_gb < 11:
        sys.exit(f'only {free_gb:.1f} GB free under {parent}; the merged model needs ~10 GB. '
                 'Delete old checkpoints or point merged_dir at /tmp.')

    print('loading the base model in fp16 on the CPU…', flush=True)
    base = AutoModelForImageTextToText.from_pretrained(cfg['model_id'], dtype=torch.float16,
                                                       device_map='cpu', low_cpu_mem_usage=True)
    model = PeftModel.from_pretrained(base, adapter)
    model = model.merge_and_unload()
    model.config.use_cache = True
    if hasattr(model.config, 'text_config'):
        model.config.text_config.use_cache = True

    shutil.rmtree(merged, ignore_errors=True)
    model.save_pretrained(merged, max_shard_size='2GB')
    AutoProcessor.from_pretrained(cfg['model_id']).save_pretrained(merged)
    # Anything the processor did not write (generation config, video preprocessor, template).
    src = cfg['model_id']
    if not os.path.isdir(src):
        src = snapshot_download(src, allow_patterns=['*.json', '*.jinja', '*.txt'],
                                ignore_patterns=['model.safetensors.index.json'])
    for name in os.listdir(src):
        if os.path.isfile(os.path.join(src, name)) and not os.path.exists(os.path.join(merged, name)):
            shutil.copy2(os.path.join(src, name), os.path.join(merged, name))

    n = sum(p.numel() for p in model.parameters())
    has_visual = any('visual' in k for k in model.state_dict())
    print(f'merged model → {merged} ({n / 1e9:.2f}B params, vision tower kept: {has_visual})')
    print('files:', sorted(os.listdir(merged)))


if __name__ == '__main__':
    main()

In [ ]:
# 6. Choose the mix, render it with Qwen3.5's chat template, tokenize once and compute the loss labels.
# Runs as a separate process so the ~1 GB train file never stays in this kernel's memory.
run(f'python3 {SCRIPTS}/prepare_data.py --config {CONFIG}', 'prepare')

In [ ]:
# 7. Smoke test: two training steps per GPU on the LONGEST prepared rows. It proves that memory fits at
# MAX_SEQ_LEN and the loss is finite, and measures the real speed, before you spend hours on the full run.
import json
run(f'torchrun --standalone --nnodes=1 --nproc_per_node={NUM_GPUS} {SCRIPTS}/train_lora.py --config {CONFIG} --smoke',
    'smoke')
smoke = json.load(open(f'{OUT_DIR}/smoke.json'))
print(json.dumps(smoke, indent=1))
need = TRAIN_ROWS * EPOCHS
fit = smoke['rows_that_fit_time_budget']
print(f'peak GPU memory {smoke["peak_gib"]} GiB of ~14.5 | {smoke["sec_per_row_all_gpus"]} s per row (longest rows)')
if fit < need:
    print(f'NOTE: about {fit} rows fit in {TIME_BUDGET_HOURS} h, but {need} are planned. Either lower TRAIN_ROWS in '
          f'cell 3 and rerun cells 3 and 6, or keep it: training stops with a checkpoint and resumes next session.')
else:
    print(f'about {fit} rows fit in the time budget; {need} are planned.')

In [ ]:
# 8. Train on both T4s: one process per GPU. Stops by itself before TIME_BUDGET_HOURS with a checkpoint.
# Rerun this cell (in this or a new session) to resume from the newest checkpoint it can find.
import glob, json, os
if HF_REPO_ID and not glob.glob(f'{OUT_DIR}/checkpoints/checkpoint-*'):
    from huggingface_hub import snapshot_download
    try:
        hub = snapshot_download(HF_REPO_ID, allow_patterns=['last-checkpoint/*'], local_dir=f'{TMP}/hub')
        if os.path.exists(f'{hub}/last-checkpoint/trainer_state.json'):
            cfg = json.load(open(CONFIG))
            cfg['resume_from'] = f'{hub}/last-checkpoint'
            json.dump(cfg, open(CONFIG, 'w'), indent=1)
            print('resuming from the Hub checkpoint', HF_REPO_ID)
    except Exception as exc:
        print('no checkpoint on the Hub yet:', str(exc)[:200])
run(f'torchrun --standalone --nnodes=1 --nproc_per_node={NUM_GPUS} {SCRIPTS}/train_lora.py --config {CONFIG}', 'train')

In [ ]:
# 9. Training status and held-out validation metrics
import json, os
path = f'{OUT_DIR}/train_status.json'
if not os.path.exists(path):
    raise RuntimeError('no train_status.json: check the output of cell 8')
status = json.load(open(path))
print('complete:', status['complete'], f'| step {status["global_step"]}/{status["max_steps"]}',
      f'| {status["hours_this_session"]} h this session')
print('eval:', json.dumps(status['eval'], indent=1))
losses = [(h['step'], round(h['loss'], 4)) for h in status['log_history'] if 'loss' in h]
evals = [(h['step'], round(h['eval_loss'], 4)) for h in status['log_history'] if 'eval_loss' in h]
print('train loss (step, loss):', losses[-12:])
print('eval loss  (step, loss):', evals)
if not status['complete']:
    print('Training is not finished: save this version (or let it push to the Hub) and rerun the notebook to resume. '
          'You can still merge the partial adapter in cell 10 to try it.')

In [ ]:
# 10. Merge a LoRA adapter into a 16-bit model for vLLM (on the CPU; about 10 GB of RAM and disk, 10-20 minutes).
# By default this merges the final adapter. To serve an earlier checkpoint instead, set MERGE_FROM to one of the
# folders listed below, run this cell, then rerun cell 11 (vLLM restarts on the new weights; the tunnel URL stays).
import glob, json, os

MERGE_FROM = None   # e.g. f'{OUT_DIR}/checkpoints/checkpoint-40'; None = the final adapter in {OUT_DIR}/adapter

print('checkpoints on disk (eval loss = the latest evaluation at or before that step):')
for ck in sorted(glob.glob(f'{OUT_DIR}/checkpoints/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1])):
    state = json.load(open(f'{ck}/trainer_state.json'))
    evals = [(h['step'], h['eval_loss']) for h in state.get('log_history', []) if 'eval_loss' in h]
    note = f' | eval loss {evals[-1][1]:.4f} at step {evals[-1][0]}' if evals else ''
    print(f'  {ck}  (step {state["global_step"]}, epoch {state.get("epoch", 0):.2f}{note})')

adapter = MERGE_FROM or f'{OUT_DIR}/adapter'
assert os.path.exists(f'{adapter}/adapter_config.json'), f'no LoRA adapter in {adapter}'
print('merging', adapter)
run(f'python3 {SCRIPTS}/merge_lora.py --config {CONFIG} --adapter {adapter}', 'merge')
assert os.path.exists(f'{MERGED_DIR}/config.json'), 'no merged model: see the output above'
json.dump({'adapter': adapter}, open(f'{MERGED_DIR}/merged_from.json', 'w'))
print('merged model now comes from', adapter)

In [ ]:
# 11. Serve the merged model with vLLM: an OpenAI-compatible API on port 8000, across both T4s.
# vLLM goes into its own virtual environment under /tmp: it brings its own torch, which would otherwise replace
# the one the training cells use.
import json, os, subprocess, sys, time, urllib.request

SERVED_NAME = 'decillion-dsh-qwen35-4b'
PORT = 8000
MAX_LEN = 20480          # dsh prompts are 6.8k-18.7k tokens, plus room for the reply. Only 8 of the 32 layers keep a
                         # KV cache (the rest are linear attention), so this fits on 2xT4 with tensor parallelism.
VLLM_SPEC = 'vllm==0.34.0'
TOOL_PARSER = 'qwen3_coder'   # Qwen3.5 writes <tool_call><function=…><parameter=…> blocks (the Qwen3-Coder format)
VENV = f'{TMP}/vllm-env'
SECRETS = f'{TMP}/secrets'
VLLM_LOG = f'{TMP}/vllm.log'
os.makedirs(SECRETS, exist_ok=True)
assert os.path.exists(f'{MERGED_DIR}/config.json'), f'no merged model at {MERGED_DIR}: run cell 10 first'
if os.path.exists(f'{MERGED_DIR}/merged_from.json'):
    print('serving the model merged from', json.load(open(f'{MERGED_DIR}/merged_from.json'))['adapter'])

if not os.path.exists(f'{VENV}/bin/vllm'):
    print('installing vLLM into', VENV, '(a few minutes)…')
    !pip install -q uv
    !uv venv -q --python {sys.executable} {VENV}
    !uv pip install -q --python {VENV}/bin/python {VLLM_SPEC}
!{VENV}/bin/python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)"

# The tunnel publishes this port to the internet, so the API always needs a key. Set a Kaggle secret
# DECILLION_API_KEY to keep it stable across sessions; otherwise a new one is generated.
API_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret('DECILLION_API_KEY')
except Exception:
    pass
API_KEY = API_KEY or globals().get('API_KEY') or 'sk-decillion-' + os.urandom(16).hex()

subprocess.run(['pkill', '-f', 'vllm.entrypoints.openai.api_server'], capture_output=True)
time.sleep(3)
cmd = [f'{VENV}/bin/python', '-m', 'vllm.entrypoints.openai.api_server',
       '--model', MERGED_DIR, '--served-model-name', SERVED_NAME,
       '--host', '127.0.0.1', '--port', str(PORT), '--api-key', API_KEY,
       '--dtype', 'float16',                   # T4 has no bf16
       '--max-model-len', str(MAX_LEN), '--tensor-parallel-size', '2',
       '--gpu-memory-utilization', '0.90', '--max-num-seqs', '4',
       '--enforce-eager',                      # no CUDA graphs: less memory pressure on T4
       '--enable-auto-tool-choice', '--tool-call-parser', TOOL_PARSER, '--reasoning-parser', 'qwen3',
       '--limit-mm-per-prompt', '{"image": 0, "video": 0}']   # text-only fine-tune: no vision memory
print('starting vLLM…')
proc = subprocess.Popen(cmd, stdout=open(VLLM_LOG, 'w'), stderr=subprocess.STDOUT)

def served_ids():
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}/v1/models', headers={'Authorization': f'Bearer {API_KEY}'})
    return [m['id'] for m in json.load(urllib.request.urlopen(req, timeout=5))['data']]

deadline, ids = time.time() + 1200, None
while time.time() < deadline and proc.poll() is None:
    try:
        ids = served_ids()
        break
    except Exception:
        time.sleep(5)
if ids != [SERVED_NAME]:
    print(open(VLLM_LOG).read()[-5000:])
    raise RuntimeError('vLLM did not come up: see its log above (on T4, check for unsupported-GPU errors from '
                       'the attention or linear-attention kernels; lowering MAX_LEN helps with out-of-memory)')

SERVE = {'base_url_local': f'http://127.0.0.1:{PORT}/v1', 'port': PORT, 'model': SERVED_NAME,
         'api_key': API_KEY, 'max_model_len': MAX_LEN, 'merged_dir': MERGED_DIR}
SERVE_JSON = f'{SECRETS}/serve.json'      # holds the key: kept out of /kaggle/working, which is saved and shareable
json.dump(SERVE, open(SERVE_JSON, 'w'), indent=1)
print('serving', SERVED_NAME, '→', SERVE['base_url_local'], '| API key saved to', SERVE_JSON)

In [ ]:
# 12. Expose the vLLM API to the internet with a Cloudflare quick tunnel (no account needed)
import json, os, re, subprocess, time, urllib.request

SERVE_JSON = f'{TMP}/secrets/serve.json'
SERVE = json.load(open(SERVE_JSON))
PORT, API_KEY = SERVE['port'], SERVE['api_key']
CF, CF_LOG = f'{TMP}/bin/cloudflared', f'{TMP}/cloudflared.log'
os.makedirs(os.path.dirname(CF), exist_ok=True)
if not os.path.exists(CF):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CF}
    !chmod +x {CF}

# Make sure the model is still up before publishing it.
req = urllib.request.Request(f'http://127.0.0.1:{PORT}/v1/models', headers={'Authorization': f'Bearer {API_KEY}'})
urllib.request.urlopen(req, timeout=10).read()

subprocess.run(['pkill', '-f', 'cloudflared tunnel'], capture_output=True)
time.sleep(2)
if os.path.exists(CF_LOG):
    os.remove(CF_LOG)
# HTTP/2 uses TCP 443 and is often more reliable in hosted notebook environments.
subprocess.Popen([CF, 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate', '--protocol', 'http2',
                  '--logfile', CF_LOG], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

url, deadline = None, time.time() + 90
while time.time() < deadline and not url:
    time.sleep(2)
    if os.path.exists(CF_LOG):
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open(CF_LOG).read())
        url = m.group(0) if m else None
if not url:
    print(open(CF_LOG).read()[-2000:] if os.path.exists(CF_LOG) else '')
    raise RuntimeError('cloudflared printed no URL')
print('public URL:', url)

def public_ids():
    req = urllib.request.Request(f'{url}/v1/models', headers={'Authorization': f'Bearer {API_KEY}'})
    return [m['id'] for m in json.load(urllib.request.urlopen(req, timeout=10))['data']]

deadline = time.time() + 120
while True:
    try:
        ids = public_ids()
        break
    except Exception as exc:
        if time.time() > deadline:
            raise RuntimeError(f'the tunnel is up but the API is not reachable yet: {exc}')
        time.sleep(3)

SERVE['base_url_public'] = url + '/v1'
json.dump(SERVE, open(SERVE_JSON, 'w'), indent=1)
print('reachable from the internet →', SERVE['base_url_public'], '| model', ids[0])
print('This URL lives only while this session and cloudflared stay up; rerun this cell and update the provider in')
print('Decillion if it changes. A named Cloudflare tunnel (dashboard → token) gives a stable hostname instead.')

In [ ]:
# 13. Test the public API end to end, the way the Decillion LLM gateway uses it
import gzip, json, os, time
import requests
from jsonschema import validate
from openai import OpenAI
from transformers import AutoTokenizer

SHOW_SECRETS = False   # True prints the API key in the output (outputs are saved with the notebook)
SERVE_JSON = f'{TMP}/secrets/serve.json'
SERVE = json.load(open(SERVE_JSON))
assert 'base_url_public' in SERVE, 'run cell 12 first: the public URL is missing'
BASE, KEY, MODEL = SERVE['base_url_public'].rstrip('/'), SERVE['api_key'], SERVE['model']
client = OpenAI(base_url=BASE, api_key=KEY, timeout=600, max_retries=1)
HEADERS = {'Authorization': f'Bearer {KEY}'}
tok = AutoTokenizer.from_pretrained(SERVE['merged_dir'])
REPLY_BUDGET = 1024

results = []
def check(name, ok, detail=''):
    results.append((name, bool(ok)))
    print(('PASS' if ok else 'FAIL'), '|', name, ('— ' + detail if detail else ''))

# 1. The OpenAI surface the gateway expects, with and without the key.
r = requests.get(f'{BASE}/models', timeout=30)
check('refuses requests without the key', r.status_code == 401, f'HTTP {r.status_code}')
r = requests.get(f'{BASE}/models', headers=HEADERS, timeout=30)
ids = [m.get('id') for m in r.json().get('data', [])] if r.status_code == 200 else []
check('/v1/models lists the fine-tuned model', ids == [MODEL], str(ids))

# 2. A plain completion.
resp = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': 'Reply with exactly: DECILLION-OK'}],
    temperature=0, max_tokens=64, extra_body={'chat_template_kwargs': {'enable_thinking': False}})
text = (resp.choices[0].message.content or '').strip()
check('a chat completion comes back', 'DECILLION-OK' in text, repr(text[:80]))

# 3. The gateway's own probe shape: streaming, with usage, timed.
t0 = time.time()
first = usage = None
streamed = reasoned = ''
body = {'model': MODEL, 'stream': True, 'stream_options': {'include_usage': True},
        'messages': [{'role': 'user', 'content': 'In about 60 words, explain why the sky is blue.'}],
        'max_tokens': 600}
with requests.post(f'{BASE}/chat/completions', json=body, stream=True, timeout=300,
                   headers={**HEADERS, 'Accept': 'text/event-stream'}) as r:
    r.raise_for_status()
    for line in r.iter_lines(decode_unicode=True):
        if not line or not line.startswith('data:'):
            continue
        payload = line[5:].strip()
        if payload == '[DONE]':
            break
        chunk = json.loads(payload)
        if chunk.get('usage'):
            usage = chunk['usage']
        for choice in chunk.get('choices', []):
            delta = choice.get('delta') or {}
            reasoning = delta.get('reasoning_content') or delta.get('reasoning') or ''
            if first is None and (delta.get('content') or reasoning or delta.get('tool_calls')):
                first = time.time() - t0
            streamed += delta.get('content') or ''
            reasoned += reasoning
check('streams tokens and reports usage for metering',
      first is not None and (streamed.strip() or reasoned.strip()) and usage
      and usage.get('prompt_tokens', 0) > 0 and usage.get('completion_tokens', 0) > 0,
      f'first token {first and round(first, 2)}s, {usage and usage.get("completion_tokens")} completion tokens')
print('   sample:', repr((streamed.strip() or reasoned.strip())[:160]))

# 4. Held-out validation rows: does the served model behave like the dsh agent?
path = f'{DIST_DIR}/validation.jsonl.gz'
if not os.path.exists(path):
    path = f'{DATA_DIR}/dist/validation.jsonl.gz'
rows = [json.loads(line) for line in gzip.open(path, 'rt', encoding='utf-8')]

def first_expected(row):
    for i, message in enumerate(row['messages']):
        if message['role'] == 'assistant':
            calls = message.get('tool_calls') or []
            return i, (calls[0]['function']['name'] if calls else None)
    return None, None

def fits(messages, tools):
    text = tok.apply_chat_template(messages, tools=tools, add_generation_prompt=True, tokenize=False)
    n = len(tok(text, add_special_tokens=False)['input_ids'])
    return n + REPLY_BUDGET <= SERVE['max_model_len']

wanted = ['dev.bugfix', 'people.ask_user', 'memory.remember', 'team.delegate_parallel']
picked = []
for category in wanted:
    for row in rows:
        ai, expected = first_expected(row)
        if row['category'] == category and expected and fits(row['messages'][:ai], row['tools']):
            picked.append((row, ai, expected))
            break
check('held-out validation rows are available', len(picked) == len(wanted), f'{len(picked)}/{len(wanted)}')

matched = 0
for row, ai, expected in picked:
    tools = row['tools']
    try:
        resp = client.chat.completions.create(model=MODEL, messages=row['messages'][:ai], tools=tools,
                                              tool_choice='auto', temperature=0, max_tokens=REPLY_BUDGET)
    except Exception as exc:
        check(f'{row["category"]}: calls {expected}', False, str(exc)[:140])
        continue
    message = resp.choices[0].message
    call = (message.tool_calls or [None])[0]
    got = call.function.name if call else None
    args_ok = False
    if call:
        schema = next((t.get('function', t).get('parameters') for t in tools
                       if t.get('function', t).get('name') == got), None)
        try:
            validate(json.loads(call.function.arguments), schema)
            args_ok = True
        except Exception:
            args_ok = False
    matched += got == expected
    check(f'{row["category"]}: calls {expected}', got == expected and args_ok, f'got {got}, arguments valid={args_ok}')
check('behaves like the dsh agent (3 of 4 held-out tool choices)', matched >= 3, f'{matched}/{len(picked)}')

# 5. Tool calls also have to stream as deltas (what dsh consumes).
if picked:
    row, ai, expected = picked[0]
    body = {'model': MODEL, 'messages': row['messages'][:ai], 'tools': row['tools'], 'tool_choice': 'auto',
            'temperature': 0, 'max_tokens': REPLY_BUDGET, 'stream': True, 'stream_options': {'include_usage': True}}
    name = arguments = ''
    with requests.post(f'{BASE}/chat/completions', json=body, stream=True, timeout=600,
                       headers={**HEADERS, 'Accept': 'text/event-stream'}) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data:'):
                continue
            payload = line[5:].strip()
            if payload == '[DONE]':
                break
            for choice in json.loads(payload).get('choices', []):
                for call in (choice.get('delta') or {}).get('tool_calls') or []:
                    fn = call.get('function') or {}
                    name += fn.get('name') or ''
                    arguments += fn.get('arguments') or ''
    check('tool calls stream as OpenAI deltas', name == expected and arguments.lstrip().startswith('{'),
          f'name={name!r} arguments={arguments[:80]!r}')

# 6. The fine-tune, not base Qwen: the dsh persona is in the weights.
system = next((m['content'] for m in rows[0]['messages'] if m['role'] == 'system'), '')
resp = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'system', 'content': system},
                           {'role': 'user', 'content': 'In one short sentence, who are you?'}],
    temperature=0, max_tokens=REPLY_BUDGET)
message = resp.choices[0].message
extra = message.model_extra or {}
reasoning = getattr(message, 'reasoning_content', None) or extra.get('reasoning_content') or extra.get('reasoning') or ''
answer = ((message.content or '') + ' ' + reasoning).strip()
check('answers as decillion, not as base Qwen', 'decillion' in answer.lower(), repr(answer[:140]))

# 7. The provider to paste into Decillion (Admin → AI → Providers → Add → Custom). It holds the key, so it is
# written under /tmp, not to the saved notebook output.
gateway = {'name': 'Decillion dsh Qwen3.5-4B (Kaggle vLLM)', 'kind': 'custom', 'base_url': BASE,
           'api_key': KEY, 'enabled': True,
           'models': [{'id': MODEL, 'name': 'Decillion dsh Qwen3.5-4B', 'context_window': SERVE['max_model_len'],
                       'capabilities': ['chat'], 'input_price': 0, 'output_price': 0}]}
provider_path = f'{TMP}/secrets/decillion_gateway_provider.json'
json.dump(gateway, open(provider_path, 'w'), indent=1)
shown = gateway if SHOW_SECRETS else {**gateway, 'api_key': KEY[:14] + '…'}
print(f'\nprovider for Decillion (full version in {provider_path}; set SHOW_SECRETS = True to print the key):')
print(json.dumps(shown, indent=1))

failed = [name for name, ok in results if not ok]
print(f'\n{len(results) - len(failed)}/{len(results)} checks passed')
if failed:
    raise AssertionError('failed: ' + '; '.join(failed))
print('The fine-tuned model is served correctly. Keep this session (and cloudflared) alive while the provider is in')
print('use; after a disconnect rerun cells 11-12 and update the base URL.')